# Stage 2c: Induction heads on TinyStories — from zero

This notebook trains a 2-layer, 2-head transformer on **TinyStories** and then
builds up the **induction circuit** one idea at a time. It is a personal
learning notebook: the goal is that by the end you can look at an attention
pattern and *see* the circuit.

**Limits up front:** the model is tiny (500-token BPE vocab, d_model=256,
context 64) and trained only on TinyStories. Its predictions are toy-grade.
That is fine — the induction circuit is the same algorithm frontier models use,
and small is what makes it visible.

## The route (5 panels, one idea each)

1. **The behavior** — the model completes a repeated pair. No mechanism yet.
2. **L0 prev-token head** — every token learns who is directly behind it.
3. **L1 induction head** — a repeated token looks at what *followed* its first occurrence.
4. **Composition** — how the two heads chain (K-composition). Why one layer can't do this.
5. **The punchline** — the same circuit fires on *random* tokens: it's an algorithm, not memorization. That is in-context learning.

Then a **sandbox**: feed the model your own text and watch both heads.

## Paper map

| Notebook section | Concept | Paper |
|---|---|---|
| Training + eval patterns (below) | repeated-random-sequence evaluation | [Olsson et al. 2022](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html), "Argument 1" |
| `induction_from_patterns` score | prefix-matching score | Olsson et al. 2022, "Definition of induction heads" |
| Panel 1 | behavioral definition of in-context learning | Olsson et al. 2022, "Defining in-context learning" |
| Panel 2 | previous-token heads, QK vs OV circuits | [Elhage et al. 2021](https://transformer-circuits.pub/2021/framework/index.html), "Two-Layer Attention-Only Models" |
| Panel 3 | induction head = prefix matching + copying | Olsson et al. 2022 |
| Panel 4 | K-composition / virtual heads; why depth ≥ 2 | Elhage et al. 2021, "Composition" + term expansion |
| Panel 5 | induction heads drive in-context learning | Olsson et al. 2022, main claim |


In [ ]:
import sys
sys.path.insert(0, '/home/yassermakram/code/fanous-llm-lens/notebooks/education')
import torch
import tiny
import numpy as np
import os
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, normalizers, pre_tokenizers, decoders

# Config
VOCAB_SIZE = 500
N_LAYERS = 2
N_HEADS = 2
D_MODEL = 256
N_CTX = 64
LR = 3e-4
BATCH_SIZE = 32
STEPS = 2000
CKPT_DIR = '/home/yassermakram/code/fanous-llm-lens/notebooks/education/checkpoints/induction_tiny'
TOK_PATH = os.path.join(CKPT_DIR, 'tokenizer.json')
MODEL_PATH = os.path.join(CKPT_DIR, 'model.pt')
CORPUS_PATH = os.path.join(CKPT_DIR, 'corpus.txt')
TOKENS_PATH = os.path.join(CKPT_DIR, 'tokens.npy')

os.makedirs(CKPT_DIR, exist_ok=True)
device = tiny.device()
print(f'[setup] device={device}')

In [ ]:
def download_tinystories(cache_path: str, max_chars: int = 5_000_000) -> str:
    if os.path.exists(cache_path):
        with open(cache_path, 'r', encoding='utf-8') as f:
            text = f.read()
        print(f'[dataset] cache hit: {len(text):,} chars')
        return text
    
    print(f'[dataset] downloading TinyStories (up to {max_chars:,} chars)...')
    ds = load_dataset('roneneldan/TinyStories', split='train')
    
    text_parts = []
    total = 0
    for item in ds:
        if total >= max_chars:
            break
        text = item['text'].strip()
        if text and len(text) > 10:
            text_parts.append(text)
            total += len(text) + 1
    
    text = '\n'.join(text_parts)
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    with open(cache_path, 'w', encoding='utf-8') as f:
        f.write(text)
    print(f'[dataset] saved {len(text):,} chars to {cache_path}')
    return text

corpus_text = download_tinystories(CORPUS_PATH)

In [ ]:
def train_tokenizer(text: str, out_path: str):
    if os.path.exists(out_path):
        print(f'[tok] cache hit: {out_path}')
        return Tokenizer.from_file(out_path)
    
    print(f'[tok] training {VOCAB_SIZE}-vocab BPE...')
    tok = Tokenizer(models.BPE(unk_token='[UNK]'))
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=['[UNK]', '[BOS]', '[EOS]']
    )
    tok.normalizer = normalizers.NFKC()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.decoder = decoders.BPEDecoder()
    
    chunk = 1_000_000
    tok.train_from_iterator(
        (text[i:i+chunk] for i in range(0, len(text), chunk)),
        trainer=trainer
    )
    tok.save(out_path)
    print(f'[tok] saved {tok.get_vocab_size()} tokens -> {out_path}')
    return tok

tokenizer = train_tokenizer(corpus_text, TOK_PATH)
vocab_size = tokenizer.get_vocab_size()
print(f'[tok] vocab={vocab_size}')

In [ ]:
def tokenize_corpus(text: str, tok, cache_path: str) -> np.ndarray:
    if os.path.exists(cache_path):
        ids = np.load(cache_path)
        print(f'[tok] cache hit: {len(ids):,} ids')
        return ids
    
    print('[tok] encoding corpus...')
    chunks = []
    step = 5_000_000
    for i in range(0, len(text), step):
        chunk_ids = tok.encode(text[i:i+step]).ids
        chunks.append(np.asarray(chunk_ids, dtype=np.uint16))
    
    ids = np.concatenate(chunks) if chunks else np.zeros(0, dtype=np.uint16)
    np.save(cache_path, ids)
    print(f'[tok] saved {len(ids):,} tokens to {cache_path}')
    return ids

corpus_ids = tokenize_corpus(corpus_text, tokenizer, TOKENS_PATH)
print(f'[tok] {len(corpus_ids):,} total tokens')

In [ ]:
if os.path.exists(MODEL_PATH):
    print(f'[train] loading checkpoint: {MODEL_PATH}')
    model = tiny.make_tiny_model(
        n_layers=N_LAYERS, n_heads=N_HEADS,
        d_vocab=vocab_size, n_ctx=N_CTX, d_model=D_MODEL,
    )
    model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
else:
    print('[train] no checkpoint found, initializing fresh model')
    model = tiny.make_tiny_model(
        n_layers=N_LAYERS, n_heads=N_HEADS,
        d_vocab=vocab_size, n_ctx=N_CTX, d_model=D_MODEL,
    )

n_params = sum(p.numel() for p in model.parameters())
print(f'[train] model params: {n_params:,}')
model.to(device)
model.train()

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)

n_seqs = len(corpus_ids) // N_CTX
data = torch.from_numpy(corpus_ids[:n_seqs * N_CTX].reshape(n_seqs, N_CTX).astype(np.int64)).to(device)
print(f'[train] {n_seqs:,} sequences of length {N_CTX}')

checkpoint_step = 500
print(f'[train] training for {STEPS} steps, checkpointing every {checkpoint_step} steps...')

for step in range(STEPS):
    idx = torch.randint(0, n_seqs, (BATCH_SIZE,))
    batch = data[idx]
    
    opt.zero_grad()
    loss = model(batch, return_type='loss')
    loss.backward()
    opt.step()
    
    if step % max(STEPS // 5, 1) == 0:
        print(f'  step {step:>4}/{STEPS}  loss={float(loss):.3f}')
    
    if (step + 1) % checkpoint_step == 0:
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  [train] checkpoint saved')

In [ ]:
import glob

if os.path.exists(MODEL_PATH):
    print(f'[load] using checkpoint: {MODEL_PATH}')
    model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
    model.to(device)
    model.eval()
else:
    print('[load] no checkpoint found, using model from training cell')

# Uncomment to clear all caches:
# for f in [TOK_PATH, MODEL_PATH, CORPUS_PATH, TOKENS_PATH]:
#     if os.path.exists(f):
#         os.remove(f)
#         print(f'[clear] removed {f}')

In [ ]:
def make_paper_patterns(n_patterns: int = 20, block_size: int = 8, max_gap: int = 20, seed: int = 42):
    rng = np.random.RandomState(seed)
    patterns = []
    
    for _ in range(n_patterns):
        block_a = rng.randint(0, vocab_size, size=block_size)
        gap_size = rng.randint(0, max_gap + 1)
        gap = rng.randint(0, vocab_size, size=gap_size)
        pattern_type = rng.choice(['gap', 'no_gap', 'multi'])
        
        if pattern_type == 'gap':
            pattern = np.concatenate([block_a, gap, block_a])
            src = np.array([0] * len(block_a) + [-1] * len(gap) + [i for i in range(len(block_a))])
        elif pattern_type == 'no_gap':
            pattern = np.concatenate([block_a, block_a])
            src = np.array([0] * len(block_a) + [i for i in range(len(block_a))])
        else:
            block_b = rng.randint(0, vocab_size, size=block_size)
            gap1 = rng.randint(0, vocab_size, size=rng.randint(0, max_gap//2 + 1))
            gap2 = rng.randint(0, vocab_size, size=rng.randint(0, max_gap//2 + 1))
            pattern = np.concatenate([block_a, gap1, block_b, gap2, block_a])
            first_a_len = len(block_a) + len(gap1) + len(block_b) + len(gap2)
            src = np.array([0] * len(block_a) + [-1] * (len(gap1) + len(block_b) + len(gap2)) + [i for i in range(len(block_a))])
        
        patterns.append({
            'pattern': pattern,
            'src': src,
            'type': pattern_type,
            'length': len(pattern)
        })
    
    return patterns

eval_patterns = make_paper_patterns(n_patterns=20, block_size=8, max_gap=20, seed=42)
print(f'[eval] generated {len(eval_patterns)} patterns')
for p in eval_patterns[:3]:
    print(f'  type={p["type"]:8s}  length={p["length"]:3d}  src_sum={p["src"].sum():3d}')

In [ ]:
def induction_from_patterns(patterns_list, src_list):
    all_scores = []
    for pats, src in zip(patterns_list, src_list):
        # pats shape: [n_layers, batch, n_heads, seq, seq]
        n_layers = pats.shape[0]
        n_heads = pats.shape[2]
        S = pats.shape[-1]
        scores_layer = torch.zeros(n_layers, n_heads)
        for L in range(n_layers):
            for h in range(n_heads):
                tot, cnt = 0.0, 0
                for t in range(S):
                    s = int(src[t])
                    if s >= 0 and s + 1 < t:
                        tot += float(pats[L, 0, h, t, s + 1])
                        cnt += 1
                scores_layer[L, h] = tot / max(cnt, 1)
        all_scores.append(scores_layer)
    return all_scores

print('[eval] running inference on all patterns...')
all_patterns_pats = []
all_patterns_src = []

for p in eval_patterns:
    toks = torch.tensor([p['pattern']]).to(device)
    _, cache = model.run_with_cache(toks)
    # Stack layers into [n_layers, n_heads, S, S]
    pats = torch.stack([cache['pattern', L].detach().cpu() for L in range(model.cfg.n_layers)])
    all_patterns_pats.append(pats)
    all_patterns_src.append(p['src'])

all_scores = induction_from_patterns(all_patterns_pats, all_patterns_src)
print('[eval] done')

In [ ]:
import pandas as pd

# Read actual layers/heads from scores tensor shape
# all_scores is a list of [n_layers, n_heads] tensors (one per pattern)
actual_n_layers = all_scores[0].shape[0]
actual_n_heads = all_scores[0].shape[1]

rows = []
for i, (scores, p) in enumerate(zip(all_scores, eval_patterns)):
    for L in range(actual_n_layers):
        for h in range(actual_n_heads):
            rows.append({
                'pattern_idx': i,
                'type': p['type'],
                'length': p['length'],
                'layer': L,
                'head': h,
                'score': float(scores[L, h])
            })

df = pd.DataFrame(rows)

print('=== Induction scores by layer ===')
summary = df.groupby(['layer', 'head'])['score'].agg(['mean', 'max', 'std']).round(3)
print(summary)
print()

print('=== Best pattern per layer ===')
for L in range(actual_n_layers):
    best_idx = df[df['layer'] == L]['score'].idxmax()
    best_row = df.loc[best_idx]
    print(f'  L{L}: pattern {best_row["pattern_idx"]} (type={best_row["type"]}, score={best_row["score"]:.3f})')
print()

In [ ]:
# Viz setup: shared helpers + pick the running example for panels 1-4
import importlib
import plotly.graph_objects as go

import induction_viz

importlib.reload(induction_viz)
from induction_viz import attn_heatmap, diag_score, pick_example, prefix_matching_score


def encode(text):
    return tokenizer.encode(text).ids


def decode_token(tid):
    return tokenizer.decode([int(tid)]).strip() or '·'


def topk5(ids):
    with torch.no_grad():
        logits = model(torch.tensor([ids]).to(device))[0, -1]
    probs = torch.softmax(logits, dim=-1)
    vals, idx = torch.topk(probs, 5)
    return [(int(i), float(v)) for i, v in zip(idx, vals)]


# Prompts end on the SECOND occurrence of a token; the picker keeps the first
# one whose continuation the model actually predicts (top-3). It fails loudly
# rather than fall back to random tokens — panels 1-4 must be meaningful text.
CANDIDATES = [
    'Tom and Lily went to the park . Tom and',
    'One day Tim found a big red ball . Tim',
    'The little bird flew up to the tree . The little bird',
    'Sara had a small cat . The cat was soft . Sara had',
    'Ben and Tom played all day . Ben and',
]
ex = pick_example(CANDIDATES, encode, decode_token, topk5)
print(f'[example] prompt: {ex.prompt!r}')
print(f'[example] tokens: {ex.tokens}')
print(f'[example] 2nd occurrence at pos {ex.query_pos} should attend pos {ex.key_pos} ({ex.target_str!r})')
print(f'[example] model top-5: {[(t, round(p, 3)) for t, p in ex.topk]}')

with torch.no_grad():
    _, ex_cache = model.run_with_cache(torch.tensor([ex.ids]).to(device))
ex_pats = torch.stack(
    [ex_cache['pattern', L].detach().cpu() for L in range(model.cfg.n_layers)]
)
S = len(ex.ids)
print(f'[example] ex_pats shape: {tuple(ex_pats.shape)}  ← (layer, batch, head, query, key)')

# Head selection: L0 by prev-token diagonal on THIS prompt, L1 by mean
# prefix-matching score over the 20 random eval patterns (the score table).
L0_head = max(range(model.cfg.n_heads), key=lambda h: diag_score(ex_pats[0, 0, h].numpy()))
L1_head = int(df[df['layer'] == 1].groupby('head')['score'].mean().idxmax())
print(f'[heads] L0 prev-token head: {L0_head}   L1 induction head: {L1_head}')


## Panel 1 — the behavior, before any mechanism

The prompt ends on the **second** occurrence of a token pair. If the model has
an induction circuit, it should predict the token that followed the **first**
occurrence — even though this tiny model has no idea what the sentence means.

**Paper hook — Olsson et al. 2022, "Defining in-context learning":** in-context
learning is *behavior*: the model gets better at predicting tokens later in the
context because it can reuse what already appeared. The bar chart below is that
behavior at a single position. The rest of the notebook opens the box to find
the mechanism.


In [ ]:
from plotly.subplots import make_subplots

first_a = ex.key_pos - 1
colors = ['#d9d9d9'] * S
colors[first_a] = '#ffcc80'      # first occurrence of A
colors[ex.key_pos] = '#a5d6a7'   # B1: the token that followed it
colors[ex.query_pos] = '#ffcc80' # second occurrence of A (the query)

fig = make_subplots(
    rows=2, cols=1, row_heights=[0.35, 0.65], vertical_spacing=0.18,
    subplot_titles=(
        'the input, token by token (orange = repeated token, green = what followed it)',
        f'model top-5 predictions after the second {ex.tokens[ex.query_pos]!r}',
    ),
)
fig.add_trace(
    go.Scatter(
        x=list(range(S)), y=[0] * S, mode='markers+text',
        text=ex.tokens, textposition='top center',
        textfont=dict(size=11, family='monospace'),
        marker=dict(size=26, color=colors, symbol='square'),
        hoverinfo='skip',
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=[S], y=[0], mode='markers+text', text=['?'], textposition='top center',
        textfont=dict(size=11, family='monospace'),
        marker=dict(size=26, color='#ffffff', symbol='square', line=dict(color='#333', width=1)),
        hoverinfo='skip',
    ),
    row=1, col=1,
)
top_labels = [t for t, _ in ex.topk]
top_probs = [p for _, p in ex.topk]
bar_colors = ['#2e7d32' if t == ex.target_str else '#90a4ae' for t in top_labels]
fig.add_trace(go.Bar(x=top_labels, y=top_probs, marker_color=bar_colors), row=2, col=1)
fig.update_xaxes(visible=False, range=[-0.6, S + 0.6], row=1, col=1)
fig.update_yaxes(visible=False, range=[-0.6, 0.9], row=1, col=1)
fig.update_yaxes(title_text='probability', range=[0, 1], row=2, col=1)
fig.update_layout(width=840, height=480, showlegend=False,
                  title_text='Panel 1 — the model completes the repeated pair')
fig.show()
print(f'[P1] target {ex.target_str!r} probability: {dict(ex.topk).get(ex.target_str, 0.0):.3f}')


## Panel 2 — the L0 prev-token head: "who is behind me?"

Layer 0 can only look at raw token embeddings — no token knows anything about
its neighbours yet. The prev-token head fixes that: **every position attends to
the position directly behind it** and copies that token's identity into its own
residual stream. On the heatmap this is the sub-diagonal (outlined). Read one
row: "this query token looks at → that key token."

**Paper hook — Elhage et al. 2021:** attention heads factor into a **QK
circuit** (*where* to look — this diagonal) and an **OV circuit** (*what* to
move — here, the identity of the previous token). Prev-token heads are the
layer-0 ingredient their two-layer analysis needs for induction. The stamp this
head writes ("Tom is behind me") is what layer 1 will search for.


In [ ]:
L0_pat = ex_pats[0, 0, L0_head].numpy()
print(f'[P2] L0_pat shape: {L0_pat.shape}  ← (query, key)')
fig = attn_heatmap(
    L0_pat, ex.tokens,
    highlight=[(q, q - 1) for q in range(1, S)],
    title=f'Panel 2 — L0 head {L0_head}: every token looks one step back',
)
fig.show()
print(f'[P2] prev-token (diagonal) strength: {diag_score(L0_pat):.2f}  (1.0 = perfect)')


## Panel 3 — the L1 induction head: "what followed me last time?"

Same axes, same tokens, layer 1. The circled cell is the whole story: the
query is the **second** occurrence of the repeated token, and it attends to the
token **after** the **first** occurrence — exactly the token the model then
predicts in Panel 1.

**Paper hook — Olsson et al. 2022, definition of induction heads:** a head is
an induction head if it does **prefix matching** (at a repeated token, attend
to the token after the previous occurrence — this circled cell) and **copying**
(its OV circuit raises the probability of the attended token). The
`induction_from_patterns` score computed earlier in this notebook is a
prefix-matching score with known source positions; `prefix_matching_score`
printed below is the same measurement on this real sentence.


In [ ]:
L1_pat = ex_pats[1, 0, L1_head].numpy()
print(f'[P3] L1_pat shape: {L1_pat.shape}  ← (query, key)')
fig = attn_heatmap(
    L1_pat, ex.tokens,
    highlight=[(ex.query_pos, ex.key_pos)],
    title=(
        f'Panel 3 — L1 head {L1_head}: the second {ex.tokens[ex.query_pos]!r} '
        f'looks at what followed the first'
    ),
)
fig.show()
print(f'[P3] attention on the circled cell: {L1_pat[ex.query_pos, ex.key_pos]:.2f}')
print(f'[P3] prefix-matching score on this prompt: {prefix_matching_score(L1_pat, ex.ids):.2f}')


## Panel 4 — how the two heads compose (schematic, not model data)

Neither head alone can do induction. The prev-token head only ever looks one
step back; the induction head needs to know *which key has the repeated token
behind it* — information that does not exist in the raw embeddings. The chain:

1. **L0** stamps "«A» is behind me" into the residual stream at B₁'s position.
2. **L1**'s query at the second «A» asks "who has «A» behind them?" — its **key**
   at B₁ is built from the L0 stamp. Query matches key → attention lands on B₁.
3. **L1**'s OV circuit copies B into the output → the prediction in Panel 1.

**Paper hook — Elhage et al. 2021, "Composition":** step 2 is
**K-composition** — the L1 head's key reads from residual content *written by
an L0 head*, forming a "virtual head" that neither layer contains alone. Their
term-expansion argument shows a **one-layer** model has no such term: this is
why induction requires depth ≥ 2. (The sibling notebook
`stage2_dash2_composition_induction_reference.ipynb` measures this same
K-composition directly from the weight matrices.)


In [ ]:
A, B = ex.tokens[ex.query_pos], ex.target_str
names = [f'{A}\u2081', f'{B}\u2081', '…', f'{A}\u2082', f'{B}?']
node_colors = ['#ffcc80', '#a5d6a7', '#eeeeee', '#ffcc80', '#ffffff']

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[0, 1, 2, 3, 4], y=[0] * 5, mode='markers+text', text=names,
        textposition='middle center', textfont=dict(size=13, family='monospace'),
        marker=dict(size=58, color=node_colors, symbol='square', line=dict(color='#555', width=1)),
        hoverinfo='skip',
    )
)


def arrow(x0, x1, y, color, label):
    # lane arrows: risers connect the lane to the two nodes it links
    for xr in (x0, x1):
        fig.add_shape(type='line', x0=xr, x1=xr, y0=0.12, y1=y,
                      line=dict(color=color, width=1, dash='dot'))
    fig.add_annotation(x=x1, y=y, ax=x0, ay=y, xref='x', yref='y', axref='x', ayref='y',
                       showarrow=True, arrowhead=2, arrowwidth=2, arrowcolor=color)
    fig.add_annotation(x=(x0 + x1) / 2, y=y + 0.13, text=label, showarrow=False,
                       font=dict(size=10, color=color))


arrow(0, 1, 0.45, '#2b6cb0', f'1· L0 prev-token head stamps "{A} is behind me" into {B}\u2081')
arrow(3, 1, 0.85, '#e65100', f'2· L1 query "who has {A} behind them?" matches the stamp — K-composition')
arrow(1, 4, 1.25, '#2e7d32', f'3· L1 OV circuit copies "{B}" into the prediction')

fig.update_xaxes(visible=False, range=[-0.6, 4.6])
fig.update_yaxes(visible=False, range=[-0.5, 1.6])
fig.update_layout(width=900, height=420,
                  title='Panel 4 — the induction circuit as a chain of two heads')
fig.show()


## Panel 5 — the punchline: it works on gibberish

Everything so far used one meaningful sentence. But the 20 evaluation patterns
scored earlier are **uniform-random token IDs** — deliberate nonsense. Below is
the same L1 head on one of them: the tick labels are gibberish, and the stripe
is still there (outlined cells = where prefix matching says it should be).

The circuit never cared about meaning. It is an **algorithm over repetition**:
*find the previous occurrence of the current token, look at what followed,
predict that*. That is why it fires on text it has never seen — and that
generalization is what Olsson et al. mean by in-context learning.

**Paper hook — Olsson et al. 2022, "Argument 1":** they evaluate induction
heads on repeated **random** sequences for exactly this reason — random tokens
rule out memorized bigrams, so anything that scores high must implement the
abstract algorithm. Their main claim: these heads are the dominant mechanism of
in-context learning in small transformers. The fresh-seed patterns in the next
cell are this notebook's held-out version of that argument.


In [ ]:
# best-scoring SHORT random pattern (≤ 24 tokens keeps tick labels readable)
short = df[(df['layer'] == 1) & (df['head'] == L1_head) & (df['length'] <= 24)]
p_idx = int(short.loc[short['score'].idxmax(), 'pattern_idx'])
p = eval_patterns[p_idx]
rand_tokens = [decode_token(t) for t in p['pattern']]
rand_L1 = all_patterns_pats[p_idx][1, 0, L1_head].numpy()
print(f'[P5] rand_L1 shape: {rand_L1.shape}  ← (query, key)')

stripe = [
    (t, int(p['src'][t]) + 1)
    for t in range(len(p['src']))
    if int(p['src'][t]) >= 0 and int(p['src'][t]) + 1 < t
]
fig = attn_heatmap(
    rand_L1, rand_tokens, highlight=stripe,
    title=(
        f'Panel 5 — same L1 head {L1_head}, RANDOM tokens '
        f'(pattern {p_idx}, type={p["type"]}): the stripe survives'
    ),
)
fig.show()
print(f'[P5] prefix-matching score on this random pattern: {float(all_scores[p_idx][1][L1_head]):.2f}')
print(f'[P5] mean over all 20 random patterns: '
      f'{df[(df["layer"] == 1) & (df["head"] == L1_head)]["score"].mean():.2f}')


In [ ]:
fresh_patterns = make_paper_patterns(n_patterns=10, block_size=8, max_gap=20, seed=999)
fresh_pats = []
fresh_srcs = []

for p in fresh_patterns:
    toks = torch.tensor([p['pattern']]).to(device)
    _, cache = model.run_with_cache(toks)
    pats = torch.stack([cache['pattern', L].detach().cpu() for L in range(model.cfg.n_layers)])
    fresh_pats.append(pats)
    fresh_srcs.append(p['src'])

fresh_scores = induction_from_patterns(fresh_pats, fresh_srcs)
fresh_df = pd.DataFrame([
    {'pattern_idx': i, 'layer': L, 'head': h, 'score': float(scores[L, h])}
    for i, scores in enumerate(fresh_scores)
    for L in range(N_LAYERS)
    for h in range(actual_n_heads)
])

print('[fresh] induction scores on unseen patterns:')
print(fresh_df.groupby(['layer', 'head'])['score'].mean().round(3))
print()
print(f'[fresh] L1 mean score: {fresh_df[fresh_df["layer"]==1]["score"].mean():.3f}')
print(f'[fresh] L0 mean score: {fresh_df[fresh_df["layer"]==0]["score"].mean():.3f}')
print(f'[fresh] L1 > L0: {fresh_df[fresh_df["layer"]==1]["score"].mean() > fresh_df[fresh_df["layer"]==0]["score"].mean()}')

## Sandbox — poke the circuit yourself

`explore(text)` runs your text through the model and shows both heads through
the same heatmap lens as panels 2–3, plus the prefix-matching score.

Things to try:
- repeat a name: `"Ben saw a dog . Ben"` — does the stripe appear?
- break the repetition: change the second `Ben` to `Sam` — stripe gone?
- widen the gap between occurrences — does the score hold? (Panel 3's cell
  should move, not fade: prefix matching is position-relative, not distance-fixed)


In [ ]:
def explore(text: str):
    ids = encode(text)
    if len(ids) < 3:
        print('[sandbox] need at least 3 tokens')
        return
    if len(ids) > N_CTX:
        ids = ids[:N_CTX]
        print(f'[sandbox] truncated to {N_CTX} tokens')
    toks = [decode_token(t) for t in ids]
    with torch.no_grad():
        _, cache = model.run_with_cache(torch.tensor([ids]).to(device))
    for L, head, name in [(0, L0_head, 'prev-token'), (1, L1_head, 'induction')]:
        pat = cache['pattern', L][0, head].detach().cpu().numpy()
        print(f'[sandbox] L{L} h{head} pat shape: {pat.shape}  ← (query, key)')
        attn_heatmap(pat, toks, title=f'L{L} head {head} — {name}').show()
    l1 = cache['pattern', 1][0, L1_head].detach().cpu().numpy()
    print(f'[sandbox] prefix-matching score (L1 h{L1_head}): {prefix_matching_score(l1, ids):.2f}')


explore('The dog saw a cat . The dog')


## Recap

1. The model **completes repeated pairs** it has never seen (Panel 1) —
   in-context learning as behavior.
2. An **L0 prev-token head** gives every token its left neighbour's identity
   (Panel 2, the diagonal).
3. An **L1 induction head** uses that stamp to attend to *what followed the
   first occurrence* (Panel 3, the circled cell / the stripe).
4. The two chain via **K-composition** — the reason this needs two layers
   (Panel 4).
5. The circuit fires on **random tokens** (Panel 5): an algorithm over
   repetition, not memorized text — Olsson et al.'s argument that induction
   heads drive in-context learning.

**Where this connects:** `stage2_dash2_composition_induction_reference.ipynb`
measures the same K-composition from the weight matrices (a 0.43 virtual-head
strength); this notebook observed it behaviorally in the attention patterns.
Two views of one circuit.
